In [1]:
import scanpy as sc
import sys
sys.path.append("/project/Wellcome_Discovery/lturiano/GENESIS/00_modules")
import data_loader
import numpy as np
import json
import torch

In [2]:
home = "/project/Wellcome_Discovery/lturiano/GENESIS/"
data = "/project/Wellcome_Discovery/datashare/lturiano/data/"

In [3]:
gex = sc.read_h5ad(data + "gex_filt_aligned.h5ad")

In [3]:
from utils import extract_ensg
ensg_vocab = extract_ensg(gex, gene_id_key="gene_id")

NameError: name 'gex' is not defined

In [13]:
ensg_vocab = extract_ensg(gex, gene_id_key="gene_id")
ensg_vocab = np.unique(ensg_vocab)
ensg_vocab = np.sort(ensg_vocab)  # deterministic

ENSG_to_idx = {g: int(i) for i, g in enumerate(ensg_vocab)}

with open(home + "ENSG_to_idx_gex.json", "w") as f:
    json.dump(ENSG_to_idx, f)

print("Saved ENSG_to_idx_gex.json with", len(ENSG_to_idx), "genes")

Saved ENSG_to_idx_gex.json with 37989 genes


In [7]:
# Try if it works
gex = sc.read_h5ad(data + "gex_filt_aligned.h5ad")

In [5]:
import json

with open(home + "ENSG_to_idx_gex.json", "r") as f:
    ENSG_to_idx = json.load(f)

n_vocab = len(ENSG_to_idx)

n_vocab

37989

In [8]:
import numpy as np
import torch

def extract_ensg(adata, gene_id_key="gene_id"):
    if gene_id_key in adata.var.columns:
        return adata.var[gene_id_key].astype(str).values
    return np.asarray(adata.var_names.astype(str))

ensg_in = extract_ensg(gex, gene_id_key="gene_id")  # genes in THIS dataset column order

keep_mask = np.array([g in ENSG_to_idx for g in ensg_in], dtype=bool)   # (G_in,)
gene_idx  = torch.tensor([ENSG_to_idx[g] for g in ensg_in[keep_mask]],
                         dtype=torch.long)                               # (G_kept,)

print("Input genes:", len(ensg_in), "kept:", keep_mask.sum())

Input genes: 37989 kept: 37989
